In [1]:
import torch
import json
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from diffusers import AutoencoderKL
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import kagglehub

/home/incor/Dataset/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
path = kagglehub.dataset_download("awsaf49/coco-2017-dataset")
print("Path to dataset files:", path)
path = "/home/incor/.cache/kagglehub/datasets/awsaf49/coco-2017-dataset/versions/2/coco2017/"

Path to dataset files: /home/incor/.cache/kagglehub/datasets/awsaf49/coco-2017-dataset/versions/2


In [33]:
from pathlib import Path

# Convert your kagglehub string to a Path object
base_path = Path(path)

# Dynamically locate the exact nested directory containing the annotations
ANNOTATION_FILE = Path("/home/incor/.cache/kagglehub/datasets/awsaf49/coco-2017-dataset/versions/2/coco2017/annotations/captions_val2017.json")

# Dynamically locate the true nested image directory by searching for the missing file
IMAGE_DIR = Path("/home/incor/.cache/kagglehub/datasets/awsaf49/coco-2017-dataset/versions/2/coco2017/val2017")

print(f"Annotations mapped to: {ANNOTATION_FILE}")
print(f"Images mapped to: {IMAGE_DIR}")

OUTPUT_DIR = Path("./coco_vae_latents_val")

Annotations mapped to: /home/incor/.cache/kagglehub/datasets/awsaf49/coco-2017-dataset/versions/2/coco2017/annotations/captions_val2017.json
Images mapped to: /home/incor/.cache/kagglehub/datasets/awsaf49/coco-2017-dataset/versions/2/coco2017/val2017


In [34]:
# 2. Hardware and Model Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [35]:
# Load the VAE in float16 to preserve VRAM on the 8GB GPU
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=torch.bfloat16)
vae = vae.to(device)
vae.eval() # Set to evaluation mode to disable gradient tracking

/home/incor/Dataset/.venv/lib/python3.11/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True, bias=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
          )
        )
      )
      (1): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affin

In [36]:
# 3. Image Preprocessing
# AutoencoderKL expects images to be strictly normalized to the [-1, 1] range
transform = transforms.Compose([
    transforms.Resize((256, 256)),      # Standardize spatial dimensions for DiT
    transforms.ToTensor(),              # Converts physical pixels to [0, 1]
    transforms.Normalize([0.5], [0.5])  # Shifts the tensor range to [-1, 1]
])

# 4. Custom Dataset to load images by their explicit COCO IDs
class COCOImageDataset(Dataset):
    def __init__(self, json_path, img_dir, transform=None):
        with open(json_path, 'r') as f:
            coco_data = json.load(f)
        
        # Extract unique image IDs and map them to their corresponding local filenames
        self.images = {img['id']: img['file_name'] for img in coco_data['images']}
        self.img_ids = list(self.images.keys())
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = self.img_dir / self.images[img_id]
        
        # Convert to RGB to ensure 3 uniform channels (stripping grayscale or alpha layers)
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return img_id, image

# A batch size of 32 consumes roughly 1.5 to 2 GB of VRAM during pure inference
dataset = COCOImageDataset(ANNOTATION_FILE, IMAGE_DIR, transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)


In [37]:
# 5. Extraction Loop
SHARD_SIZE = 10000
current_shard = {}
shard_idx = 0

with torch.no_grad():
    for batch_ids, batch_images in tqdm(dataloader, desc="Extracting VAE Latents"):
        batch_images = batch_images.to(device, dtype=torch.bfloat16)
        
        # Extract the continuous latent distributions and sample discrete vectors
        latent_dist = vae.encode(batch_images).latent_dist
        latents = latent_dist.sample()
        
        # CRITICAL: Scale latents by the VAE's scaling factor to enforce unit variance
        latents = latents * vae.config.scaling_factor
        
        for img_id, latent in zip(batch_ids, latents):
            # Move back to CPU and cast to float32 to ensure stable, universally compatible storage
            current_shard[img_id.item()] = latent.cpu().to(torch.float32)
            
        if len(current_shard) >= SHARD_SIZE:
            shard_file = OUTPUT_DIR / f"vae_latents_shard_{shard_idx:03d}.pt"
            torch.save(current_shard, shard_file)
            shard_idx += 1
            current_shard = {}

# Save the final, incomplete shard to capture the remainder of the dataset
if len(current_shard) > 0:
    shard_file = OUTPUT_DIR / f"vae_latents_shard_{shard_idx:03d}.pt"
    torch.save(current_shard, shard_file)
    shard_idx += 1

print(f"\nDone! Successfully saved {shard_idx} VAE latent shards to {OUTPUT_DIR}")

Extracting VAE Latents: 100%|██████████| 157/157 [01:00<00:00,  2.58it/s]



Done! Successfully saved 1 VAE latent shards to coco_vae_latents_val
